# Love Island: Download Reddit Data
This notebook walks through the downloading of Reddit data.

## Collecting URLs
I compiled Love Island Reddit URL's manually. It was the only way I could get around no self-serve Reddit API. I took the Top 4 reddit pages for each episode (more or less).

```
site:reddit.com/r/LoveIslandUSA 'Season 7 episode 15'
```

For season 8, I took the top 5 pages that had to do with the episode. Sometimes, the finale was mixed in there so I had to be careful.

```
site:reddit.com/r/loveislandusa 'season 8 episode 31'
```

Then, have to download using the following.

In [2]:
import re              # EXTRACT SUBMISSION IDS
import subprocess      # RUN BDFR
import time            # TRACK TIME AND SLEEP
import pandas as pd

from pathlib import Path
from datetime import datetime

In [3]:
# SETTINGS

# root      = '../reddit-download/s7/'
# url_file  = Path(root + 's7_reddit_urls.txt')
# out_dir   = Path('../data/LoveIslandUSA_S7')
# log_file  = Path(root + 's7_bdfr_download_log.csv')
# temp_file = Path(root + 'current_url.txt')

root      = '../reddit-download/s8/'
url_file  = Path(root + 's8_reddit_urls.txt')
out_dir   = Path('../data/LoveIslandUSA_S8')
log_file  = Path(root + 's8_bdfr_download_log.csv')
temp_file = Path(root + 'current_url.txt')

min_wait_sec = 120
wait_multiplier = 1.5

out_dir.mkdir(
    exist_ok=True
)

In [4]:
# READ URLS
urls = [
    x.strip()
    for x in url_file.read_text().splitlines()
    if x.strip()
]

# REMOVE DUPLICATES BUT KEEP ORDER
urls = list(
    dict.fromkeys(urls)
)

# HOW MANY URLS?
print(
    f'N URLs: {len(urls)}'
)

N URLs: 177


In [5]:
# HELPERS
def extract_submission_id(url):
    # PULL THE REDDIT THREAD ID
    match = re.search(
        r'/comments/([a-z0-9]+)/',
        url
    )

    if match:
        return match.group(1)

    return None


def get_existing_log():
    # CHECK IF LOG FILE EXISTS AND READ IT INTO A DATAFRAME
    if log_file.exists():

        return pd.read_csv(
            log_file
        )

    return pd.DataFrame()

In [6]:
# SKIP THE URLS THAT HAVE ALREADY BEEN SUCCESSFULLY DOWNLOADED
existing_log_df = get_existing_log()

if not existing_log_df.empty and 'success' in existing_log_df.columns:

    completed_urls = set(
        existing_log_df
        .loc[
            existing_log_df['success'] == True,
            'url'
        ]
    )

else:

    completed_urls = set()

urls_to_run = [
    url
    for url in urls
    if url not in completed_urls
]


print(
    f'Already successful: {len(completed_urls)}'
)

print(
    f'Remaining to run: {len(urls_to_run)}'
)

Already successful: 0
Remaining to run: 177


In [7]:
# LOOP THROUGH URLS
for i, url in enumerate(urls_to_run, start=1):

    # GET SUBMISSION ID
    submission_id = extract_submission_id(
        url
    )

    # PRINT PROGRESS
    print('\n' + '=' * 100)
    print(
        f'{i}/{len(urls_to_run)}'
    )
    print(
        f'ID: {submission_id}'
    )
    print(
        url
    )

    # WRITE CURRENT URL
    temp_file.write_text(
        url + '\n'
    )

    # TRACK START TIME
    start_dt = datetime.now()
    start = time.time()

    # RUN BDFR
    result = subprocess.run(
        [
            'bdfr',
            'clone',
            str(out_dir),
            '--include-id-file',
            str(temp_file)
        ],
        capture_output=True,
        text=True
    )

    # CALCULATE ELAPSED TIME
    elapsed = time.time() - start
    end_dt = datetime.now()

    # SAVE TERMINAL OUTPUT
    stdout = result.stdout or ''
    stderr = result.stderr or ''
    stderr_lower = stderr.lower()

    # CHECK WHETHER DOWNLOAD SUCCEEDED
    success = (
        result.returncode == 0
        and
        'failed to be cloned' not in stderr_lower
        and
        'received 429' not in stderr_lower
    )

    # CHECK FOR RATE LIMITING
    got_429 = (
        'received 429' in stderr_lower
        or
        'too many requests' in stderr_lower
    )

    print(
        f'{elapsed/60:.1f} min | Success: {success}'
    )

    if got_429:
        print(
            'Got 429 / rate limited'
        )

    # SAVE DOWNLOAD LOG
    new_log = pd.DataFrame([
        {
            'run_start': start_dt,
            'run_end': end_dt,
            'url': url,
            'submission_id': submission_id,
            'returncode': result.returncode,
            'success': success,
            'got_429': got_429,
            'elapsed_sec': elapsed,
            'stdout_tail': stdout[-1000:],
            'stderr_tail': stderr[-2000:]
        }
    ])

    # APPEND TO EXISTING LOG
    if log_file.exists():

        old_log = pd.read_csv(
            log_file
        )

        log_df = pd.concat(
            [
                old_log,
                new_log
            ],
            ignore_index=True
        )

    else:

        log_df = new_log

    # SAVE LOG
    log_df.to_csv(
        log_file,
        index=False
    )

    # CALCULATE WAIT TIME
    wait_time = max(
        min_wait_sec,
        elapsed * wait_multiplier
    )

    # WAIT LONGER AFTER 429S
    if got_429:

        wait_time = max(
            wait_time,
            10 * 60
        )

    print(
        f'Sleeping {wait_time/60:.1f} minutes...'
    )

    # PAUSE BEFORE NEXT THREAD
    time.sleep(
        wait_time
    )


1/177
ID: 1tvbdoe
https://www.reddit.com/r/LoveIslandUSA/comments/1tvbdoe/season_8_episode_1_post_episode_discussion/
9.7 min | Success: True
Sleeping 14.5 minutes...

2/177
ID: 1tvbdpi
https://www.reddit.com/r/LoveIslandUSA/comments/1tvbdpi/season_8_episode_1_cast_opinions_discussion/
5.2 min | Success: True
Sleeping 7.7 minutes...

3/177
ID: 1tv942p
https://www.reddit.com/r/LoveIslandUSA/comments/1tv942p/spoiler_season_8_episode_1_tuesday_june_02_830_pm/
2.7 min | Success: True
Sleeping 4.1 minutes...

4/177
ID: 1tvh1f9
https://www.reddit.com/r/LoveIslandUSA/comments/1tvh1f9/season_8_episode_1_post_show_thoughts_and_first/
0.2 min | Success: True
Sleeping 2.0 minutes...

5/177
ID: 1tv9sq9
https://www.reddit.com/r/LoveIslandUSA/comments/1tv9sq9/without_ads_season_8_episode_1_tuesday_june_02_9/
3.7 min | Success: True
Sleeping 5.5 minutes...

6/177
ID: 1tv9sql
https://www.reddit.com/r/LoveIslandUSA/comments/1tv9sql/with_ads_season_8_episode_1_tuesday_june_02_9_pm/
4.5 min | Success: T

These data can now be found in `data/LoveIslandUSA_SX`.

## Further Data Curation
More text data can be found on the LoveIslandUSA Reddit.